# Routenspezifische Fine-Tuning-Datasets

Routet Train und Validation durch `f_small -> r1 -> f_mid -> r2 -> f_large`.
Train erhält 30% Replay, Validation kein Replay; Testdaten und neue Rejector-Targets
werden nicht verwendet. Gespeichert werden kompakte Index-Datasets mit Labels,
Metadaten, Provenance und Summary.

## 1. Setup und Konfiguration

Alle veränderbaren Pfade, Thresholds und Laufparameter stehen am Anfang.
`REPLAY_FRACTION=0.30` bezeichnet den Anteil im finalen Trainingsdataset.

In [2]:
from __future__ import annotations

from datetime import datetime, timezone
import json
import math
from pathlib import Path
import random
from typing import Any
import warnings

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from pathlib import Path
import sys

ROOT = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import training
import training_models
from DMTimeShardDataset import DMTimeShardDataset
from embedding_processing_models import build_embedding_processing


DATA_ROOT = ROOT.parent / "DM_time_dataset_creator" / "outputs"

# Explizite Quellpfade; Testpfade sind absichtlich nicht Teil der Config.
TRAIN_DATA_DIR = DATA_ROOT / "dm_time_shards" / "train"
VAL_DATA_DIR = DATA_ROOT / "dm_time_shards" / "val"
DATASET_PREFIX = "B0531+21_59000_48386"
DATASET_CFG = {
    "output_dir": str(DATA_ROOT),
    "prefix": DATASET_PREFIX,
}

CHECKPOINTS = {
    "f_small": ROOT / "final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_241002_3_GAP-014-0.764-0.740.pth",
    "f_mid": ROOT / "final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_241002_5_GAP-060-0.973-0.948.pth",
    "f_large": ROOT / "final_checkpoints/baseline_rejection_ensemble/prot-DM_time_binary_classificator_resnet18-003-0.993-0.993.pth",
    "r1": ROOT / "final_checkpoints/baseline_rejection_ensemble/prot-run_embedding_3_GAP_conv_mlp_lr1.05e-05_wd0.00e+00_drop0.0_channels64_extraFalse_pool7_hidden64_worker2_trial3-042-0.712-0.635.pth",
    "r2": ROOT / "final_checkpoints/baseline_rejection_ensemble/prot-run_embedding_r2_conv_mlp_lr4.51e-05_wd0.00e+00_drop0.2_channels64_extraTrue_pool7_hidden128_worker13_trial6-035-0.825-0.842.pth",
}

# Bereits festgelegte statische Reject-Score-Thresholds.
# Reject-Regel: softmax(logits)[:, 1] >= threshold.
R1_THRESHOLD = 0.548808
R2_THRESHOLD = 0.381863

R1_KWARGS = {
    "model_name": "conv_mlp",
    "cnn_channels": 64,
    "extra_conv": False,
    "pool_size": 7,
    "hidden_dim": 64,
    "dropout": 0.0,
}
R2_KWARGS = {
    "model_name": "conv_mlp",
    "cnn_channels": 64,
    "extra_conv": True,
    "pool_size": 7,
    "hidden_dim": 128,
    "dropout": 0.2,
}

REPLAY_FRACTION = 0.0 #0.30
SEED = 20260625
OUTPUT_DIR = ROOT / "finetune" / "finetune_datasets_NOREPLAY"
BATCH_SIZE = 1024
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

META_COLUMNS = ["snr", "t_min", "t_max", "dm", "label_raw"]
MODEL_ORDER = ("f_small", "f_mid", "f_large")
OUTPUT_STEMS = {
    "f_small": {"train": "f_small_train_finetune", "val": "f_small_val_route"},
    "f_mid": {"train": "f_mid_train_finetune", "val": "f_mid_val_route"},
    "f_large": {"train": "f_large_train_finetune", "val": "f_large_val_route"},
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR}")

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def checkpoint_descriptor(path: Path) -> dict[str, Any]:
    stat = path.stat()
    return {
        "path": str(path.resolve()),
        "size_bytes": int(stat.st_size),
        "mtime_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
    }


if not 0.0 <= REPLAY_FRACTION < 1.0:
    raise ValueError("REPLAY_FRACTION must be in [0, 1).")

for source_path in (TRAIN_DATA_DIR, VAL_DATA_DIR):
    if not source_path.is_dir():
        raise FileNotFoundError(source_path)

for checkpoint_name, checkpoint_path in CHECKPOINTS.items():
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Missing {checkpoint_name} checkpoint: {checkpoint_path}")

seed_everything(SEED)
print("Configuration validated.")


Device: cuda
Output directory: /cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/finetune/finetune_datasets_NOREPLAY
Configuration validated.


## 2. Modelle und Quelldaten laden

In [3]:
def extract_state_dict(checkpoint: Any) -> dict[str, torch.Tensor]:
    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, dict):
                return value
    if not isinstance(checkpoint, dict):
        raise TypeError(f"Unsupported checkpoint type: {type(checkpoint)!r}")
    return checkpoint


def get_rejector_state_dict(checkpoint: Any) -> dict[str, torch.Tensor]:
    """Entfernt Wrapper-Präfixe aus bestehenden EmbeddingRejector-Checkpoints."""
    state_dict = extract_state_dict(checkpoint)
    clean: dict[str, torch.Tensor] = {}
    for key, value in state_dict.items():
        if key.startswith("1.net."):
            clean["net." + key[len("1.net."):]] = value
        elif key.startswith("1."):
            clean[key[len("1."):]] = value
        elif key.startswith("embedding_processing."):
            clean[key[len("embedding_processing."):]] = value
    return clean or state_dict


def load_classifier(model_key: str, mode: str, checkpoint_path: Path) -> torch.nn.Module:
    model = training_models.models_htable[model_key](
        256, mode=mode, dropout=False, device=DEVICE
    ).to(DEVICE)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(extract_state_dict(checkpoint))
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def load_rejector(
    checkpoint_path: Path,
    kwargs: dict[str, Any],
) -> tuple[torch.nn.Module, str]:
    model, feature_source = build_embedding_processing(in_channels=12, **kwargs)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(get_rejector_state_dict(checkpoint))
    model = model.to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model, feature_source


f_small = load_classifier(
    "DM_time_binary_classificator_241002_3_GAP", "dmt", CHECKPOINTS["f_small"]
)
f_mid = load_classifier(
    "DM_time_binary_classificator_241002_5_GAP", "ft", CHECKPOINTS["f_mid"]
)
f_large = load_classifier(
    "DM_time_binary_classificator_resnet18", "dmft", CHECKPOINTS["f_large"]
)
r1, r1_feature_source = load_rejector(CHECKPOINTS["r1"], R1_KWARGS)
r2, r2_feature_source = load_rejector(CHECKPOINTS["r2"], R2_KWARGS)

# f_large wird geladen und dokumentiert, aber nicht unnötig ausgeführt:
# R2-Rejects definieren die f_large-Route bereits vollständig.
print("Loaded f_small, f_mid, f_large, r1 and r2.")

class DatasetWithIndex(Dataset):
    """Ergänzt den stabilen Index des zugrunde liegenden Splits pro Sample."""

    def __init__(self, dataset: Dataset):
        self.dataset = dataset

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        sample = dict(self.dataset[index])
        sample["sample_index"] = torch.tensor(index, dtype=torch.long)
        return sample


def make_source_dataset(split: str) -> DMTimeShardDataset:
    dataset = DMTimeShardDataset(DATASET_CFG, use_freq_time=True, split=split)
    # Targets bleiben die normalen Klassifikationslabels.
    dataset.labels = training.label_encoding(dataset.labels.astype(object))
    return dataset


def make_routing_loader(dataset: Dataset) -> DataLoader:
    return DataLoader(
        DatasetWithIndex(dataset),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )


train_dataset = make_source_dataset("train")
val_dataset = make_source_dataset("val")
train_loader = make_routing_loader(train_dataset)
val_loader = make_routing_loader(val_dataset)

print(f"Train samples:      {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")


Loaded f_small, f_mid, f_large, r1 and r2.
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Train samples:      1,377,792
Validation samples: 172,032


## 3. Routing

R2 wird nur auf R1-Rejects ausgeführt. Die nächste Zelle definiert die Logik;
die darauffolgende Zelle startet den vollständigen Train-/Validation-Lauf.

In [4]:
def classifier_and_rejector_logits(
    classifier: torch.nn.Module,
    rejector: torch.nn.Module,
    feature_source: str,
    batch: dict[str, torch.Tensor],
) -> tuple[torch.Tensor, torch.Tensor]:
    model_input = classifier.features(batch)
    if model_input.ndim == 3:
        model_input = model_input.unsqueeze(0)
    classifier_logits = classifier.classifier(model_input)

    rejector_features = getattr(classifier, feature_source)(model_input)
    if rejector_features.ndim == 3:
        rejector_features = rejector_features.unsqueeze(0)
    elif feature_source == "pooled_features" and rejector_features.ndim == 1:
        rejector_features = rejector_features.unsqueeze(0)
    rejector_logits = rejector(rejector_features)
    return classifier_logits, rejector_logits


def subset_batch(
    batch: dict[str, torch.Tensor],
    cpu_mask: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """Subset aller batch-ausgerichteten Tensoren; Mask bleibt bewusst auf CPU."""
    if cpu_mask.device.type != "cpu":
        cpu_mask = cpu_mask.cpu()
    return {
        key: value[cpu_mask]
        for key, value in batch.items()
        if torch.is_tensor(value) and value.shape[0] == cpu_mask.shape[0]
    }


def route_split(
    loader: DataLoader,
    split_name: str,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    for model in (f_small, f_mid, f_large, r1, r2):
        model.eval()

    route_parts = {model_name: [] for model_name in MODEL_ORDER}
    r1_score_parts: list[np.ndarray] = []
    r2_score_parts: list[np.ndarray] = []
    seen_index_parts: list[np.ndarray] = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Routing {split_name}"):
            sample_indices = batch["sample_index"].cpu().numpy()
            seen_index_parts.append(sample_indices)

            _, r1_logits = classifier_and_rejector_logits(
                f_small, r1, r1_feature_source, batch
            )
            r1_scores = torch.softmax(r1_logits, dim=1)[:, 1]
            r1_rejected = r1_scores >= R1_THRESHOLD
            r1_rejected_cpu = r1_rejected.cpu()

            route_parts["f_small"].append(sample_indices[~r1_rejected_cpu.numpy()])
            batch_r2_scores = np.full(len(sample_indices), np.nan, dtype=np.float32)

            if r1_rejected.any():
                r2_batch = subset_batch(batch, r1_rejected_cpu)
                r2_input_indices = sample_indices[r1_rejected_cpu.numpy()]

                _, r2_logits = classifier_and_rejector_logits(
                    f_mid, r2, r2_feature_source, r2_batch
                )
                r2_scores = torch.softmax(r2_logits, dim=1)[:, 1]
                r2_rejected = r2_scores >= R2_THRESHOLD
                r2_rejected_cpu = r2_rejected.cpu().numpy()

                route_parts["f_mid"].append(r2_input_indices[~r2_rejected_cpu])
                route_parts["f_large"].append(r2_input_indices[r2_rejected_cpu])
                batch_r2_scores[r1_rejected_cpu.numpy()] = (
                    r2_scores.detach().cpu().numpy().astype(np.float32, copy=False)
                )

            r1_score_parts.append(
                r1_scores.detach().cpu().numpy().astype(np.float32, copy=False)
            )
            r2_score_parts.append(batch_r2_scores)

    routes = {
        model_name: (
            np.concatenate(parts).astype(np.int64, copy=False)
            if parts else np.empty(0, dtype=np.int64)
        )
        for model_name, parts in route_parts.items()
    }
    seen_indices = np.concatenate(seen_index_parts).astype(np.int64, copy=False)
    scores = {
        "sample_indices": seen_indices,
        "r1_scores": np.concatenate(r1_score_parts),
        "r2_scores": np.concatenate(r2_score_parts),
    }

    expected_indices = np.arange(len(loader.dataset), dtype=np.int64)
    routed_indices = np.concatenate([routes[name] for name in MODEL_ORDER])
    assert np.array_equal(seen_indices, expected_indices)
    assert len(routed_indices) == len(expected_indices)
    assert np.array_equal(np.sort(routed_indices), expected_indices)
    assert sum(len(routes[name]) for name in MODEL_ORDER) == len(loader.dataset)

    print(
        f"{split_name}: "
        + ", ".join(f"{name}={len(routes[name]):,}" for name in MODEL_ORDER)
    )
    return routes, scores


In [5]:
train_routes, train_routing_scores = route_split(train_loader, "train")
val_routes, val_routing_scores = route_split(val_loader, "val")

np.savez_compressed(
    OUTPUT_DIR / "train_routing_scores.npz",
    **train_routing_scores,
    r1_threshold=np.float64(R1_THRESHOLD),
    r2_threshold=np.float64(R2_THRESHOLD),
)
np.savez_compressed(
    OUTPUT_DIR / "val_routing_scores.npz",
    **val_routing_scores,
    r1_threshold=np.float64(R1_THRESHOLD),
    r2_threshold=np.float64(R2_THRESHOLD),
)


Routing train:   0%|          | 0/1346 [00:00<?, ?it/s]

Routing train: 100%|██████████| 1346/1346 [11:51<00:00,  1.89it/s]


train: f_small=959,490, f_mid=295,501, f_large=122,801


Routing val: 100%|██████████| 168/168 [01:48<00:00,  1.54it/s]


val: f_small=120,422, f_mid=36,127, f_large=15,483


## 4. Replay und finale Trainingsindizes

In [6]:
def replay_count_for_route(route_count: int, replay_fraction: float) -> int:
    if route_count == 0 or replay_fraction == 0.0:
        return 0
    return int(round(route_count * replay_fraction / (1.0 - replay_fraction)))


def make_finetune_indices(
    route_indices: np.ndarray,
    full_train_size: int,
    replay_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_replay = replay_count_for_route(len(route_indices), replay_fraction)
    if n_replay > full_train_size:
        raise ValueError(
            f"Need {n_replay:,} replay samples, but train split has only "
            f"{full_train_size:,}. Reduce REPLAY_FRACTION or allow replacement."
        )

    rng = np.random.default_rng(seed)
    replay_indices = rng.choice(
        full_train_size, size=n_replay, replace=False
    ).astype(np.int64, copy=False)

    combined_indices = np.concatenate([route_indices, replay_indices])
    is_replay = np.concatenate([
        np.zeros(len(route_indices), dtype=np.uint8),
        np.ones(len(replay_indices), dtype=np.uint8),
    ])
    permutation = rng.permutation(len(combined_indices))
    return combined_indices[permutation], is_replay[permutation], replay_indices


train_finetune: dict[str, dict[str, np.ndarray]] = {}
for model_number, model_name in enumerate(MODEL_ORDER, start=1):
    combined_indices, is_replay, replay_indices = make_finetune_indices(
        route_indices=train_routes[model_name],
        full_train_size=len(train_dataset),
        replay_fraction=REPLAY_FRACTION,
        seed=SEED + model_number,
    )
    train_finetune[model_name] = {
        "indices": combined_indices,
        "is_replay": is_replay,
        "replay_indices": replay_indices,
    }
    actual_replay_fraction = (
        float(is_replay.mean()) if len(is_replay) else float("nan")
    )
    print(
        f"{model_name}: route={len(train_routes[model_name]):,}, "
        f"replay={len(replay_indices):,}, final={len(combined_indices):,}, "
        f"replay share={actual_replay_fraction:.2%}"
    )


f_small: route=959,490, replay=0, final=959,490, replay share=0.00%
f_mid: route=295,501, replay=0, final=295,501, replay share=0.00%
f_large: route=122,801, replay=0, final=122,801, replay share=0.00%


## 5. Speichern

Die `.npz`-Dateien referenzieren den bestehenden `DMTimeShardDataset`, statt große
Bildtensoren zu kopieren. `load_index_dataset(...)` lädt sie später direkt.

In [7]:
class IndexedFineTuneDataset(Dataset):
    """Dataset-View auf einen bestehenden Split; unterstützt Replay-Duplikate."""

    def __init__(
        self,
        base_dataset: Dataset,
        indices: np.ndarray,
        is_replay: np.ndarray | None = None,
    ):
        self.base_dataset = base_dataset
        self.indices = np.asarray(indices, dtype=np.int64)
        if is_replay is None:
            is_replay = np.zeros(len(self.indices), dtype=np.uint8)
        self.is_replay = np.asarray(is_replay, dtype=np.uint8)
        if len(self.indices) != len(self.is_replay):
            raise ValueError("indices and is_replay must have equal length")

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        source_index = int(self.indices[index])
        sample = dict(self.base_dataset[source_index])
        sample["source_index"] = torch.tensor(source_index, dtype=torch.long)
        sample["is_replay"] = torch.tensor(
            bool(self.is_replay[index]), dtype=torch.bool
        )
        return sample


def load_index_dataset(
    dataset_file: str | Path,
    dataset_cfg: dict[str, Any] = DATASET_CFG,
    use_freq_time: bool = True,
) -> IndexedFineTuneDataset:
    """Lädt einen der erzeugten `.npz`-Outputs direkt fürs Fine-Tuning."""
    dataset_file = Path(dataset_file)
    saved = np.load(dataset_file, allow_pickle=False)
    split = str(saved["source_split"].item())
    base_dataset = DMTimeShardDataset(
        dataset_cfg, use_freq_time=use_freq_time, split=split
    )
    base_dataset.labels = training.label_encoding(
        base_dataset.labels.astype(object)
    )
    return IndexedFineTuneDataset(
        base_dataset=base_dataset,
        indices=saved["indices"],
        is_replay=saved["is_replay"],
    )


# Beispiel für später:
# f_small_train_dataset = load_index_dataset(
#     OUTPUT_DIR / "f_small_train_finetune.npz"
# )
# f_small_train_loader = DataLoader(
#     f_small_train_dataset, batch_size=256, shuffle=True, num_workers=2
# )

def json_ready(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def save_index_dataset(
    stem: Path,
    source_dataset: DMTimeShardDataset,
    source_split: str,
    indices: np.ndarray,
    is_replay: np.ndarray,
    route_count: int,
    replay_count: int,
    model_name: str,
    sampling_seed: int | None,
) -> dict[str, Path]:
    indices = np.asarray(indices, dtype=np.int64)
    is_replay = np.asarray(is_replay, dtype=np.uint8)
    if len(indices) != len(is_replay):
        raise ValueError(f"Length mismatch for {stem.name}")
    if len(indices) and (indices.min() < 0 or indices.max() >= len(source_dataset)):
        raise IndexError(f"Out-of-range source index in {stem.name}")

    dataset_path = stem.with_suffix(".npz")
    labels_path = stem.parent / f"{stem.name}_labels.npy"
    metadata_path = stem.parent / f"{stem.name}_metadata.npy"
    manifest_path = stem.parent / f"{stem.name}_metadata.json"

    labels = np.asarray(source_dataset.labels)[indices]
    metadata = np.asarray(source_dataset.metadata)[indices]

    np.savez_compressed(
        dataset_path,
        indices=indices,
        is_replay=is_replay,
        source_split=np.asarray(source_split),
        source_dataset_size=np.int64(len(source_dataset)),
        route_count=np.int64(route_count),
        replay_count=np.int64(replay_count),
        generation_seed=np.int64(SEED),
        sampling_seed=np.int64(sampling_seed if sampling_seed is not None else -1),
    )
    np.save(labels_path, labels)
    np.save(metadata_path, metadata)

    manifest = {
        "schema_version": 1,
        "dataset_name": stem.name,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "purpose": f"route-specific classifier fine-tuning for {model_name}",
        "target_semantics": "original binary classification labels; no rejector targets",
        "source": {
            "dataset_cfg": DATASET_CFG,
            "split": source_split,
            "dataset_size": len(source_dataset),
            "indices_are_relative_to_split": True,
            "test_data_used": False,
        },
        "routing": {
            "route_count": int(route_count),
            "replay_count": int(replay_count),
            "final_count": int(len(indices)),
            "r1_threshold": R1_THRESHOLD,
            "r2_threshold": R2_THRESHOLD,
            "reject_rule": "softmax(logits)[:, 1] >= threshold",
        },
        "replay": {
            "enabled": source_split == "train" and replay_count > 0,
            "requested_final_fraction": (
                REPLAY_FRACTION if source_split == "train" else 0.0
            ),
            "sampling": "uniform without stratification and without replacement",
            "seed": sampling_seed,
        },
        "files": {
            "index_dataset": dataset_path.name,
            "labels": labels_path.name,
            "metadata": metadata_path.name,
        },
        "metadata_columns": META_COLUMNS,
        "checkpoints": {
            name: checkpoint_descriptor(path)
            for name, path in CHECKPOINTS.items()
        },
    }
    manifest_path.write_text(
        json.dumps(json_ready(manifest), indent=2), encoding="utf-8"
    )
    return {
        "dataset": dataset_path,
        "labels": labels_path,
        "metadata": metadata_path,
        "manifest": manifest_path,
    }


saved_files: dict[str, dict[str, Path]] = {}

for model_name in MODEL_ORDER:
    train_stem = OUTPUT_DIR / OUTPUT_STEMS[model_name]["train"]
    train_info = train_finetune[model_name]
    saved_files[f"{model_name}_train_finetune"] = save_index_dataset(
        stem=train_stem,
        source_dataset=train_dataset,
        source_split="train",
        indices=train_info["indices"],
        is_replay=train_info["is_replay"],
        route_count=len(train_routes[model_name]),
        replay_count=len(train_info["replay_indices"]),
        model_name=model_name,
        sampling_seed=SEED + MODEL_ORDER.index(model_name) + 1,
    )

    val_stem = OUTPUT_DIR / OUTPUT_STEMS[model_name]["val"]
    val_indices = val_routes[model_name]
    saved_files[f"{model_name}_val_route"] = save_index_dataset(
        stem=val_stem,
        source_dataset=val_dataset,
        source_split="val",
        indices=val_indices,
        is_replay=np.zeros(len(val_indices), dtype=np.uint8),
        route_count=len(val_indices),
        replay_count=0,
        model_name=model_name,
        sampling_seed=None,
    )

print(f"Saved {len(saved_files)} datasets.")


Saved 6 datasets.


## 6. Summary und Konsistenzchecks

In [8]:
def numeric_stats(values: np.ndarray, prefix: str) -> dict[str, float]:
    values = np.asarray(values, dtype=np.float64)
    finite = values[np.isfinite(values)]
    if len(finite) == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_std": np.nan,
        }
    return {
        f"{prefix}_mean": float(np.mean(finite)),
        f"{prefix}_median": float(np.median(finite)),
        f"{prefix}_std": float(np.std(finite)),
    }


def summary_row(
    dataset_name: str,
    model_name: str,
    split: str,
    source_dataset: DMTimeShardDataset,
    indices: np.ndarray,
    is_replay: np.ndarray,
    route_count: int,
) -> dict[str, Any]:
    indices = np.asarray(indices, dtype=np.int64)
    is_replay = np.asarray(is_replay, dtype=np.uint8)
    labels = np.asarray(source_dataset.labels)[indices]
    metadata = np.asarray(source_dataset.metadata)[indices]
    replay_count = int(is_replay.sum())
    final_count = len(indices)

    row: dict[str, Any] = {
        "dataset": dataset_name,
        "model": model_name,
        "split": split,
        "source_split_count": len(source_dataset),
        "route_count": int(route_count),
        "route_fraction_of_source_split": (
            route_count / len(source_dataset) if len(source_dataset) else np.nan
        ),
        "replay_count": replay_count,
        "final_count": final_count,
        "route_fraction_of_final": (
            (final_count - replay_count) / final_count if final_count else np.nan
        ),
        "replay_fraction_of_final": (
            replay_count / final_count if final_count else np.nan
        ),
    }

    unique_labels, counts = np.unique(labels, return_counts=True)
    for label, count in zip(unique_labels, counts):
        label_key = str(int(label)) if float(label).is_integer() else str(label)
        row[f"label_{label_key}_count"] = int(count)
        row[f"label_{label_key}_fraction"] = (
            int(count) / final_count if final_count else np.nan
        )

    if metadata.ndim == 2 and metadata.shape[1] > 0:
        row.update(numeric_stats(metadata[:, 0], "snr"))
    else:
        row.update(numeric_stats(np.asarray([]), "snr"))

    if metadata.ndim == 2 and metadata.shape[1] > 3:
        row.update(numeric_stats(metadata[:, 3], "dm"))
    else:
        row.update(numeric_stats(np.asarray([]), "dm"))
    return row


summary_rows: list[dict[str, Any]] = []
for model_name in MODEL_ORDER:
    train_info = train_finetune[model_name]
    summary_rows.append(summary_row(
        dataset_name=f"{model_name}_train_finetune",
        model_name=model_name,
        split="train",
        source_dataset=train_dataset,
        indices=train_info["indices"],
        is_replay=train_info["is_replay"],
        route_count=len(train_routes[model_name]),
    ))

    val_indices = val_routes[model_name]
    summary_rows.append(summary_row(
        dataset_name=f"{model_name}_val_route",
        model_name=model_name,
        split="val",
        source_dataset=val_dataset,
        indices=val_indices,
        is_replay=np.zeros(len(val_indices), dtype=np.uint8),
        route_count=len(val_indices),
    ))

summary = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "finetune_dataset_summary.csv"
summary.to_csv(summary_path, index=False)

run_metadata = {
    "schema_version": 1,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_cfg": DATASET_CFG,
    "train_data_dir": TRAIN_DATA_DIR,
    "val_data_dir": VAL_DATA_DIR,
    "thresholds": {"r1": R1_THRESHOLD, "r2": R2_THRESHOLD},
    "replay_fraction": REPLAY_FRACTION,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "device": str(DEVICE),
    "test_data_used": False,
    "validation_replay_used": False,
    "targets": "unchanged classification labels",
    "summary_file": summary_path.name,
    "checkpoints": {
        name: checkpoint_descriptor(path) for name, path in CHECKPOINTS.items()
    },
}
(OUTPUT_DIR / "finetune_dataset_generation_metadata.json").write_text(
    json.dumps(json_ready(run_metadata), indent=2), encoding="utf-8"
)

display(summary)
print(f"Summary saved to {summary_path}")

required_dataset_names = [
    *(f"{model_name}_train_finetune" for model_name in MODEL_ORDER),
    *(f"{model_name}_val_route" for model_name in MODEL_ORDER),
]

for dataset_name in required_dataset_names:
    paths = saved_files[dataset_name]
    for path in paths.values():
        assert path.is_file(), path

    saved = np.load(paths["dataset"], allow_pickle=False)
    assert len(saved["indices"]) == len(saved["is_replay"])
    if dataset_name.endswith("_val_route"):
        assert int(saved["is_replay"].sum()) == 0
        assert int(saved["replay_count"]) == 0

assert summary["final_count"].ge(0).all()
assert (OUTPUT_DIR / "finetune_dataset_summary.csv").is_file()

print("All outputs and invariants validated.")
print("No test data were loaded. Validation contains no replay.")


,dataset,model,split,source_split_count,route_count,route_fraction_of_source_split,replay_count,final_count,route_fraction_of_final,replay_fraction_of_final,label_0_count,label_0_fraction,label_1_count,label_1_fraction,snr_mean,snr_median,snr_std,dm_mean,dm_median,dm_std
0,f_small_train_finetune,f_small,train,1377792,959490,0.696397,0,959490,1.0,0.0,685670,0.714619,273820,0.285381,6.561680,4.7,5.098993,80.938129,57.0,73.280564
1,f_small_val_route,f_small,val,172032,120422,0.699998,0,120422,1.0,0.0,79790,0.662587,40632,0.337413,7.185290,4.7,8.279827,83.460688,57.0,71.269825
2,f_mid_train_finetune,f_mid,train,1377792,295501,0.214474,0,295501,1.0,0.0,189644,0.641771,105857,0.358229,4.719135,4.5,1.449333,82.287417,57.0,70.648573
3,f_mid_val_route,f_mid,val,172032,36127,0.210002,0,36127,1.0,0.0,25096,0.694661,11031,0.305339,4.885977,4.5,3.424354,81.655272,57.0,73.205056
4,f_large_train_finetune,f_large,train,1377792,122801,0.089129,0,122801,1.0,0.0,28878,0.235161,93923,0.764839,4.695977,4.5,1.444128,70.649718,57.0,42.437877
5,f_large_val_route,f_large,val,172032,15483,0.090001,0,15483,1.0,0.0,3658,0.236259,11825,0.763741,4.956938,4.4,3.871708,70.640896,57.0,42.327086


Summary saved to /cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/finetune/finetune_datasets_NOREPLAY/finetune_dataset_summary.csv
All outputs and invariants validated.
No test data were loaded. Validation contains no replay.


## Verwendung beim Fine-Tuning

```python
train_ft = load_index_dataset(OUTPUT_DIR / "f_mid_train_finetune.npz")
val_route = load_index_dataset(OUTPUT_DIR / "f_mid_val_route.npz")

train_loader_ft = DataLoader(
    train_ft, batch_size=256, shuffle=True, num_workers=2
)
val_loader_route = DataLoader(
    val_route, batch_size=256, shuffle=False, num_workers=2
)
```

`sample["label"]` ist weiterhin das ursprüngliche Klassifikationstarget.